In [0]:
%sql
USE CATALOG gold_dev;
USE SCHEMA analytics;


In [0]:
%sql
CREATE OR REPLACE VIEW vw_customer_value AS
SELECT
    d.year,
    d.month,

    c.customer_key,
    c.customer_name,
    c.region,

    SUM(f.sales_amount)  AS total_sales,
    SUM(f.profit_amount) AS total_profit,
    COUNT(DISTINCT f.order_id) AS total_unique_orders,
    COUNT(*) AS total_orders_entries,
    SUM(f.order_quantity)      AS total_quantity,

    -- Average Order Value
    ROUND(
        SUM(f.sales_amount) / NULLIF(COUNT(DISTINCT f.order_id), 0),
        2
    ) AS avg_order_value,

    -- Ranking per Year
    ROW_NUMBER() OVER (
        PARTITION BY d.year
        ORDER BY SUM(f.sales_amount) DESC
    ) AS rn

FROM silver_dev.global_mart_retail.fact_sales f

JOIN silver_dev.global_mart_retail.dim_customer c
  ON f.customer_key = c.customer_key
 AND c.is_current_record = true

JOIN silver_dev.global_mart_retail.dim_date d
  ON f.order_date_key = d.date_key

GROUP BY
    d.year,
    d.month,
    c.customer_key,
    c.customer_name,
    c.region;
